# Validating Default Values (`validate_default`)

As discussed in previous sections, a major "gotcha" in Pydantic is that **it does not validate default values by default**. It assumes that if you hardcoded a default value into the schema, you (the developer) know what you are doing.

## 1. The Danger of Implicit Defaults

If you make a mistake in your schema definition, Pydantic's default behavior will allow bad state to enter your application silently.

```python
from pydantic import BaseModel

class BadModel(BaseModel):
    # DANGEROUS: The type hint requires an int, but the default is None
    field_1: int = None 
    # DANGEROUS: The type hint requires a string, but the default is an int
    field_2: str = 100 

# Pydantic skips validation, allowing this to succeed!
m1 = BadModel()
print(m1.model_dump()) 
# {'field_1': None, 'field_2': 100} -> Your types are completely broken!

```

If you had explicitly passed those values (`BadModel(field_1=None, field_2=100)`), Pydantic would have thrown a `ValidationError`. But because they fell back to the defaults, Pydantic skipped the check.

## 2. Enforcing Validation on Defaults

You can force Pydantic to validate your defaults during instantiation by setting `validate_default=True` inside the `model_config`.

```python
from pydantic import ConfigDict, ValidationError

class StrictDefaultModel(BaseModel):
    model_config = ConfigDict(validate_default=True)
    
    field_1: int = None 
    field_2: str = 100 

try:
    m2 = StrictDefaultModel()
except ValidationError as e:
    print(e)
    """
    Output:
    2 validation errors for StrictDefaultModel
    field_1: Input should be a valid integer
    field_2: Input should be a valid string
    """

```

By enabling this setting, Pydantic runs its standard validation pipeline on the default values, instantly catching the developer's mistake and preventing the model from initializing with broken state.

## 3. Why isn't this enabled by default?

1. **Performance:** Running validation checks takes compute time. If a developer provides a valid default (e.g., `radius: int = 5`), forcing Pydantic to validate that `5` is an integer every single time the model is instantiated is a waste of CPU cycles.
2. **Data Transformation:** Later in the course, you will learn how to write custom `@field_validator` functions that actively modify incoming data (e.g., automatically lowercasing an email address, or converting a string timestamp into a `datetime` object). Sometimes you *want* your hardcoded default value to be pushed through that transformation pipeline, and enabling `validate_default=True` is how you achieve that.

### Interview Preparation: Validating Defaults

<details>
<summary><b>1. Why does Pydantic skip validation on default values by default?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
Primarily for performance optimization. Pydantic assumes that if a developer explicitly hardcoded a default value into the schema (e.g., `age: int = 18`), the developer has already ensured it matches the type contract. Skipping the validation check on defaults saves CPU cycles during model instantiation, which adds up significantly in high-throughput API environments.
</details>

<details>
<summary><b>2. You review a PR and see `model_config = ConfigDict(validate_default=True)`. Besides catching developer typos, what is an architectural reason a senior engineer might intentionally enable this setting?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
They likely have custom data validators or transformers attached to the model fields. <br><br>
For example, imagine a field that accepts a raw string but uses a custom validator to automatically convert that string into an encrypted hash. If the field falls back to a plaintext default value, and `validate_default=False`, the default value will bypass the validator and remain unencrypted in the model instance. By enabling `validate_default=True`, the engineer forces the default value to run through the custom validator pipeline, ensuring the data is properly transformed before the object is created.
</details>

<details>
<summary><b>3. Look at this model: <br>`class Model(BaseModel):`<br>&nbsp;&nbsp;&nbsp;&nbsp;`field: int = "100"`<br><br>If `validate_default=True` is set on this model, and you instantiate it via `Model()`, what is the resulting type of `field`? Does it raise an error?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
It will NOT raise an error, and the resulting type will be a Python `int` (`100`). <br><br>
Because `validate_default=True` is enabled, the default string `"100"` is pushed through Pydantic's standard validation pipeline. Because Pydantic operates in Lax Coercion mode by default, the validator safely casts the string `"100"` into the integer `100`. 
</details>

<details>
<summary><b>4. If you had the exact same model from Question 3, but you also added `strict=True` to the `model_config`, what would happen?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
In that case, instantiation would fail and raise a `ValidationError`. <br><br>
Because `validate_default=True` pushes the default string `"100"` into the validator, and `strict=True` disables type coercion (forbidding string-to-int conversions), Pydantic would throw an error stating that the input should be a valid integer.
</details>